
# **MIT Academy of Engineering (MIT AOE)**
## **Department of Artificial Intelligence**
### **Assignment No. 1 – Feature Engineering on Bank Marketing  Dataset**
##Problem Type: Classification


| Name | Roll Number |
|------|--------------|
| Mit Jangid | 202401110072 |

---


1. Data Exploration and Loading


In [3]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif

# --- Configuration ---
# CHANGE THIS LINE if your file name is different (e.g., 'bank-additional-full.csv')
file_path = 'bank-additional/bank-additional-full.csv'
TARGET_COL = 'y'
K_FEATURES = 8

# --- 1. Load Data, Handle 'unknown', and Target Conversion ---
try:
    # The dataset uses a semicolon separator
    df = pd.read_csv(file_path, sep=';')
    print(f"✅ Data loaded successfully from: {file_path}")
except FileNotFoundError:
    print(f"❌ FATAL ERROR: File not found at '{file_path}'. Please check file name/path.")
    # Stop execution if the file cannot be found
    raise

# Convert the binary target 'y' ('yes'/'no') to 1/0
df[TARGET_COL] = df[TARGET_COL].map({'yes': 1, 'no': 0})

# Replace the common textual missing indicator 'unknown' with standard NaN
df.replace('unknown', np.nan, inplace=True)

print("\n--- Initial Prep Summary ---")
print(df.head())
print(f"Target variable unique values: {df[TARGET_COL].unique()}")
print(f"Missing values (NaN) per column:\n{df.isna().sum().sort_values(ascending=False).head()}")

✅ Data loaded successfully from: bank-additional/bank-additional-full.csv

--- Initial Prep Summary ---
   age        job  marital    education default housing loan    contact month  \
0   56  housemaid  married     basic.4y      no      no   no  telephone   may   
1   57   services  married  high.school     NaN      no   no  telephone   may   
2   37   services  married  high.school      no     yes   no  telephone   may   
3   40     admin.  married     basic.6y      no      no   no  telephone   may   
4   56   services  married  high.school      no      no  yes  telephone   may   

  day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0         mon  ...         1    999         0  nonexistent          1.1   
1         mon  ...         1    999         0  nonexistent          1.1   
2         mon  ...         1    999         0  nonexistent          1.1   
3         mon  ...         1    999         0  nonexistent          1.1   
4         mon  ...         1    99

2. Missing Data Imputation and Encoding

In [4]:
# --- 2. Missing Data Imputation and Encoding ---
feature_cols = df.columns.drop(TARGET_COL)
num_cols = df[feature_cols].select_dtypes(include=['int64', 'float64']).columns
cat_cols = df[feature_cols].select_dtypes(include=['object']).columns

# Impute Numerical Columns (Median)
if not num_cols.empty:
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Impute Categorical Columns (Most Frequent)
if not cat_cols.empty:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    # Imputing returns a numpy array; convert it back to a DataFrame for clean integration
    df[cat_cols] = pd.DataFrame(cat_imputer.fit_transform(df[cat_cols]), columns=cat_cols, index=df.index)

# Encoding (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df_encoded.columns = df_encoded.columns.astype(str)

print("✅ Step 2: Imputation and Encoding Complete.")
print(f"New DataFrame shape (features created): {df_encoded.shape}")
print("\n--- Encoded Feature Sample ---")
print(df_encoded.iloc[:, 15:20].head()) # Show some new encoded columns

✅ Step 2: Imputation and Encoding Complete.
New DataFrame shape (features created): (41188, 48)

--- Encoded Feature Sample ---
   job_retired  job_self-employed  job_services  job_student  job_technician
0        False              False         False        False           False
1        False              False          True        False           False
2        False              False          True        False           False
3        False              False         False        False           False
4        False              False          True        False           False


3. Scaling, PCA, and Feature Selection

In [5]:
# --- 3. Scaling, PCA, and Feature Selection ---
X = df_encoded.drop(TARGET_COL, axis=1)
y = df_encoded[TARGET_COL]

# Scaling Features (StandardScaler)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("✅ Step 3: Feature Scaling Complete.")

# PCA (Dimensionality Reduction to 2 Components)
pca = PCA(n_components=2)
pca_features = pca.fit_transform(scaled_df)
pca_df = pd.DataFrame(pca_features, columns=['PC1', 'PC2'])

print("✅ Step 4: PCA Complete.")
print(f"PCA Head:\n{pca_df.head()}")

# Feature Selection (SelectKBest using f_classif)
selector = SelectKBest(score_func=f_classif, k=K_FEATURES)
# Fit on the scaled feature data (scaled_df) and the target (y)
selector.fit(scaled_df, y)

# Get the names of the selected features
selected_features = scaled_df.columns[selector.get_support()]

print("\n✅ Step 5: Feature Selection Complete.")
print(f"Top {K_FEATURES} Selected Features:")
print(list(selected_features))

✅ Step 3: Feature Scaling Complete.
✅ Step 4: PCA Complete.
PCA Head:
        PC1       PC2
0  1.609999 -2.138651
1  1.543933 -1.051196
2  1.474435 -0.195552
3  1.622441 -1.414153
4  1.529214 -0.974291

✅ Step 5: Feature Selection Complete.
Top 8 Selected Features:
['duration', 'pdays', 'previous', 'emp.var.rate', 'euribor3m', 'nr.employed', 'poutcome_nonexistent', 'poutcome_success']


#conclusion
The entire data processing sequence has successfully prepared the Bank Marketing Dataset for machine learning model training. This preparation involved crucial steps that addressed data quality and feature complexity.

First, data cleaning was completed by converting the binary target variable to a numerical format (0/1) and handling the missing values—specifically the common 'unknown' strings—through imputation using the median for numerical features and the mode (most frequent) for categorical features. Next, all nominal categorical features were encoded using One-Hot Encoding, which expanded the feature space but made the data usable by algorithms. Finally, the data was scaled using the StandardScaler to ensure all features had a zero mean and unit variance, preventing features with larger magnitudes from dominating model training.

For optimization, two key methods were applied: PCA reduced the data to two principal components for quick structural analysis, and the SelectKBest method statistically identified and isolated the top 8 most predictive features relative to client subscription. In conclusion, the data is now a clean, optimized, and reduced set of features, ready to be directly input into a supervised classification algorithm to predict the client's decision to subscribe to a term deposit.